# Maximum-Likelihood Estimation for a Gaussian

Wiki reference for [MLE for a Gaussian](https://ml-viz-ruby.vercel.app/wiki/mle-gaussian).

**The idea in one sentence.** Maximizing the Gaussian likelihood gives closed-form estimators —
$\hat\mu$ is the sample mean and $\hat\sigma^2$ is the mean squared deviation (dividing by $n$) —
but that variance estimator is **biased low**, which is exactly why the unbiased sample variance
divides by $n-1$ (Bessel's correction).

We derive the MLE, ascend the log-likelihood, and measure the bias from scratch, **validate the
estimators and gradient ascent**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

np.random.seed(42)

## 1 — MLE formulas

From the 8-step derivation:
$$\hat{\mu}_\text{MLE} = \bar{x}, \qquad \hat{\sigma}^2_\text{MLE} = \frac{1}{n}\sum_i(x_i - \bar{x})^2$$

In [ ]:
# Wiki worked example
x = np.array([2, 4, 4, 4, 5, 5, 7, 9], dtype=float)

mu_mle    = x.mean()
sigma2_mle    = ((x - mu_mle)**2).mean()          # divisor n
sigma2_unbiased = ((x - mu_mle)**2).sum() / (len(x) - 1)  # divisor n-1

print(f"n = {len(x)}")
print(f"MLE mean:              {mu_mle:.3f}  (true=5)")
print(f"MLE variance (n):      {sigma2_mle:.3f}  (true=4)")
print(f"Unbiased variance (n-1): {sigma2_unbiased:.3f}")
print(f"Bias = {sigma2_mle - 4:.4f}  (expected: {-4/len(x):.4f})")  # E[bias] = -σ²/n

### Validate: the MLE formulas

The Gaussian MLE mean is the sample mean, and the MLE variance divides the squared deviations by
$n$ — making it **smaller** than the unbiased ($n-1$) estimator by exactly a factor
$\frac{n-1}{n}$. We confirm.

In [ ]:
print(f'MLE mean = {mu_mle:.3f}; MLE var (/n) = {sigma2_mle:.3f}; unbiased (/n-1) = {sigma2_unbiased:.3f}')
assert np.isclose(mu_mle, x.mean()), 'the MLE mean is the sample mean'
assert sigma2_mle < sigma2_unbiased, 'the MLE variance (divide by n) is smaller than the unbiased one (n-1)'
assert np.isclose(sigma2_mle, sigma2_unbiased * (len(x) - 1) / len(x)), 'MLE var = (n-1)/n * unbiased var'
print('\n✅ closed-form MLE: sample mean and (biased) mean-squared-deviation variance')

## 2 — Log-likelihood surface

The log-likelihood is a concave function of $(\mu, \sigma^2)$, peaked at the MLE.

In [ ]:
mu_grid = np.linspace(2.5, 7.5, 120)
v_grid  = np.linspace(0.5, 9.0, 120)    # v = sigma^2
MU, V = np.meshgrid(mu_grid, v_grid)

LL = np.zeros_like(MU)
for xi in x:
    LL += -0.5 * np.log(2 * np.pi * V) - (xi - MU)**2 / (2 * V)

fig, ax = plt.subplots(figsize=(8, 6))
cf = ax.contourf(MU, V, LL, levels=30, cmap='viridis')
plt.colorbar(cf, ax=ax, label='log-likelihood')
ax.scatter([mu_mle], [sigma2_mle], color='#ef4444', s=120, zorder=5,
           label=f'MLE: μ={mu_mle:.1f}, σ²={sigma2_mle:.1f}')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$\sigma^2$')
ax.set_title('Gaussian log-likelihood surface')
ax.legend()
plt.tight_layout()
plt.show()

## 3 — Gradient ascent converging to MLE

We can also find MLE by gradient ascent on $\ell(\mu, \sigma^2)$.

In [ ]:
def log_likelihood(mu, v, x):
    return np.sum(-0.5 * np.log(2 * np.pi * v) - (x - mu)**2 / (2 * v))

def grad_ll(mu, v, x):
    n = len(x)
    d_mu = np.sum(x - mu) / v
    d_v  = -n / (2 * v) + np.sum((x - mu)**2) / (2 * v**2)
    return d_mu, d_v

# Start far from the MLE
mu_curr, v_curr = 3.0, 1.0
lr = 0.03
path = [(mu_curr, v_curr, log_likelihood(mu_curr, v_curr, x))]

for _ in range(200):
    gm, gv = grad_ll(mu_curr, v_curr, x)
    mu_curr += lr * gm
    v_curr  += lr * gv
    v_curr   = max(v_curr, 0.01)   # keep variance positive
    path.append((mu_curr, v_curr, log_likelihood(mu_curr, v_curr, x)))

path = np.array(path)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Trajectory on log-likelihood surface
ax1.contourf(MU, V, LL, levels=25, cmap='viridis', alpha=0.8)
ax1.plot(path[:, 0], path[:, 1], 'w-o', markersize=2, linewidth=1, label='GD path')
ax1.scatter([mu_mle], [sigma2_mle], color='#ef4444', s=120, zorder=5, label='MLE')
ax1.set_xlabel(r'$\mu$'); ax1.set_ylabel(r'$\sigma^2$')
ax1.set_title('Gradient ascent trajectory')
ax1.legend(fontsize=9)

# Log-likelihood over iterations
ax2.plot(path[:, 2], color='#6366f1')
ax2.axhline(log_likelihood(mu_mle, sigma2_mle, x), color='#ef4444',
            linestyle='--', label='MLE optimum')
ax2.set_xlabel('Iteration'); ax2.set_ylabel('log-likelihood')
ax2.set_title('Convergence to MLE')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final: μ={mu_curr:.4f}, σ²={v_curr:.4f}")
print(f"True MLE: μ={mu_mle:.4f}, σ²={sigma2_mle:.4f}")

### Validate: gradient ascent recovers the closed-form MLE

Starting far from the optimum, gradient ascent on the log-likelihood should climb to the same
$(\hat\mu, \hat\sigma^2)$ the closed form gives. We confirm the ascent path converges to the
MLE.

In [ ]:
mu_end, v_end = path[-1][0], path[-1][1]
print(f'gradient-ascent end: mu={mu_end:.3f}, var={v_end:.3f}  vs closed-form ({mu_mle:.3f}, {sigma2_mle:.3f})')
assert abs(mu_end - mu_mle) < 0.1, 'gradient ascent recovers the MLE mean'
assert abs(v_end - sigma2_mle) < 0.5, 'gradient ascent approaches the MLE variance'
print('\n✅ the closed form and iterative optimization agree — both maximize the likelihood')

## 4 — Bias of the MLE variance as n grows

The MLE variance $\hat{\sigma}^2 = \frac{1}{n}\sum(x_i-\bar{x})^2$ underestimates
by a factor of $\frac{n-1}{n}$. We verify this empirically.

In [ ]:
TRUE_MU, TRUE_SIGMA2 = 5.0, 4.0
n_values = np.arange(3, 501)
n_trials = 2000

mle_biases = []
for n in n_values:
    samples = np.random.normal(TRUE_MU, np.sqrt(TRUE_SIGMA2), (n_trials, n))
    mle_vars = samples.var(axis=1, ddof=0)   # divisor n
    mle_biases.append(mle_vars.mean() - TRUE_SIGMA2)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_values, mle_biases, color='#6366f1', label='Empirical bias')
ax.plot(n_values, -TRUE_SIGMA2 / n_values, color='#f97316', linestyle='--',
        label=r'Theory: $-\sigma^2/n$')
ax.axhline(0, color='#555', linestyle=':')
ax.set_xlabel('Sample size n')
ax.set_ylabel('Bias of MLE variance')
ax.set_title('MLE variance bias vs. sample size')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **biased variance** | MLE divides by n, underestimates (demo) — use n-1 if unbiasedness matters |
| **small samples** | the bias is large for small n |
| **MLE vs MAP** | a prior (MAP) regularizes small-sample estimates |
| **outliers** | Gaussian MLE is not robust — a few outliers shift $\hat\mu,\hat\sigma^2$ |
| **wrong model** | MLE assumes Gaussianity; check it |

Demo: the MLE variance is biased low, worst for small n.

In [ ]:
# The subtlety MLE hides: the maximum-likelihood variance is BIASED LOW. It measures spread
# around the SAMPLE mean (which is itself fit to the data), so it systematically underestimates
# the true variance — badly for small n, vanishing as n grows. This is why the UNBIASED estimator
# divides by n-1 (Bessel's correction). We confirm the negative bias and its decay.
print(f'empirical bias of MLE variance: n=3 -> {mle_biases[0]:.3f}  (theory -sigma^2/n = {-4.0/3:.3f})')
print(f'                                n=500 -> {mle_biases[-1]:.3f}')
assert mle_biases[0] < -0.5, 'the MLE variance underestimates the truth (biased low) for small n'
assert abs(mle_biases[-1]) < 0.2, 'the bias vanishes as n grows'
print('\nThe MLE variance divides by n and is biased low -> divide by n-1 (Bessel) for an unbiased estimate.')

## ✏️ Your turn

### Exercise 1 — MLE vs. MAP

Add a Gaussian prior $\mu \sim \mathcal{N}(0, \tau^2)$ on the mean.
Derive the MAP estimate (maximizing prior × likelihood) and implement it.
Show that the MAP estimate shrinks toward 0 compared to the MLE,
and the shrinkage increases as $\tau \to 0$.

In [ ]:
# TODO(you): MAP estimate for Gaussian mean with Gaussian prior
# The MAP estimate is: mu_MAP = (n/sigma^2 * x_bar) / (n/sigma^2 + 1/tau^2)
#                             = (tau^2 * n * x_bar) / (n*tau^2 + sigma^2)

x = np.array([2, 4, 4, 4, 5, 5, 7, 9], dtype=float)
sigma2_known = 4.0  # pretend we know the true variance

# for tau in [10.0, 2.0, 1.0, 0.5]:
#     mu_map = # TODO
#     print(f"tau={tau:.1f}: mu_MAP={mu_map:.3f}  (MLE={x.mean():.3f})")  

### Exercise 2 — MLE for K=3 Gaussians (GMM)

Generate data from a 3-component Gaussian mixture and fit each component
separately assuming you know the true assignments. Compare your MLE per component
to the ground-truth parameters.

<details>
<summary>Solution outline</summary>

```python
# True params
mus = [-3, 0, 4]
sigmas = [1, 1.5, 0.8]
pis = [0.3, 0.4, 0.3]

# Generate data with known labels
n = 300
labels = np.random.choice(3, n, p=pis)
x = np.array([np.random.normal(mus[k], sigmas[k]) for k in labels])

# MLE per component: just compute mean and std of each cluster
for k in range(3):
    mask = labels == k
    print(f"k={k}: n={mask.sum()}, mu_hat={x[mask].mean():.2f}, sigma_hat={x[mask].std():.2f}")
```
</details>

## Key takeaways

- **$\hat\mu$ = sample mean, $\hat\sigma^2$ = mean squared deviation** (verified).
- **Gradient ascent recovers the closed form** (verified) — same likelihood maximum.
- **The MLE variance is biased low** (demo) — it fits spread around the fitted mean.
- **Bessel's $n-1$ correction** makes it unbiased; the difference vanishes as $n \to \infty$.